# Buffered Polygons

One polygon that safely contains a cell and everything inside it.

An H3 cell's own outline is a plain hexagon, but its descendants tile a slightly wobbly
shape that crosses that hexagon in both directions. Using the hexagon as a filter would
wrongly exclude cells that genuinely belong to the parent. Buffering fixes that.

This notebook compares the three functions on one map and checks what each actually
contains. See the [Buffered Polygons](https://khoshkhah.github.io/h3-boundary/buffering.html)
docs for the concepts.

In [ ]:
import time

import h3
import folium
from shapely.geometry import shape, Point

import h3_boundary as h3b

# Plain function names work on any install; the *_cpp variants exist only when
# the extension was compiled. use_convex_hull is C++-only, so guard for it.
print("backend:", h3b.get_backend(), "| C++ geometry:", h3b.cpp_geom_available())

cell = h3.latlng_to_cell(37.7759, -122.4180, 6)
INTERMEDIATE = 10   # same setting the docs table uses
print(f"cell {cell} (res {h3.get_resolution(cell)})")

## 1. The three polygons

- `get_buffered_boundary_polygon` — traces the real boundary, merges it, buffers it (accurate)
- the same with `use_convex_hull=True` — a hull instead of the union (fast, generous)
- `get_buffered_h3_polygon` — buffers only the cell's own hexagon (cheapest)

In [ ]:
def timed(label, fn):
    fn()
    start = time.perf_counter()
    for _ in range(10):
        out = fn()
    ms = (time.perf_counter() - start) * 1000 / 10
    verts = len(out["geometry"]["coordinates"][0])
    print(f"{label:34s} {ms:6.2f} ms   {verts:5d} vertices   "
          f"buffer {out['properties']['buffer_meters']:.1f} m")
    return out

exact    = h3b.cell_boundary_from_children(cell, target_res=INTERMEDIATE)   # no buffer, for reference
union    = timed("boundary polygon (union)",
                 lambda: h3b.get_buffered_boundary_polygon(cell, intermediate_res=INTERMEDIATE))
simple   = timed("cell buffer (no children)",
                 lambda: h3b.get_buffered_h3_polygon(cell))

hull = None
if h3b.cpp_geom_available():
    hull = timed("boundary polygon (convex hull)",
                 lambda: h3b.get_buffered_boundary_polygon_cpp(cell, INTERMEDIATE, None, True))
else:
    print("convex-hull mode needs the C++ extension — skipping")

## 2. What each one actually contains

The point of buffering is containment, so test it directly: does every vertex of every
descendant fall inside the polygon?

In [ ]:
FINE = 11
children = h3.cell_to_children(cell, FINE)
print(f"testing {len(children):,} descendants at resolution {FINE}\n")

def outside_count(feature):
    poly = shape(feature["geometry"])
    return sum(
        1 for k in children
        if any(not poly.contains(Point(lng, lat)) for lat, lng in h3.cell_to_boundary(k))
    )

exact_area = shape(exact["geometry"]).area
cases = [("boundary polygon (union)", union), ("cell buffer (no children)", simple)]
if hull is not None:
    cases.insert(1, ("boundary polygon (convex hull)", hull))

for label, feat in cases:
    missed = outside_count(feat)
    ratio = shape(feat["geometry"]).area / exact_area
    verdict = "contains everything" if missed == 0 else f"{missed:,} left OUTSIDE"
    print(f"{label:34s} area {ratio:.2f}x exact   {verdict}")

`get_buffered_h3_polygon` leaves a few percent of the descendants outside — it buffers the
cell's nominal hexagon, which the fractal boundary reaches past. It answers *roughly where
is this cell*, not *does this cell contain that point*.

The hull's cost is area rather than misses: it contains everything, but covers noticeably
more ground than the cell occupies.

## 3. On a map

Exact boundary in blue, the buffered variants over it.

In [ ]:
m = folium.Map(tiles="CartoDB positron")

exact_layer = folium.GeoJson(
    exact,
    style_function=lambda x: {"color": "#1f6feb", "weight": 2, "fillOpacity": 0.12},
    name=f"exact boundary (res {INTERMEDIATE})",
)
exact_layer.add_to(m)

folium.GeoJson(
    h3b.cell_boundary_to_geojson(cell),
    style_function=lambda x: {"color": "#238636", "weight": 2, "fillOpacity": 0,
                              "dashArray": "10,5"},
    name="the cell's own hexagon",
).add_to(m)

folium.GeoJson(
    union,
    style_function=lambda x: {"color": "#f85149", "weight": 2, "fillOpacity": 0.05,
                              "dashArray": "5,5"},
    name="buffered (union) — contains everything",
).add_to(m)

if hull is not None:
    folium.GeoJson(
        hull,
        style_function=lambda x: {"color": "#a371f7", "weight": 2, "fillOpacity": 0.05},
        name="buffered (convex hull) — fast, generous",
    ).add_to(m)

folium.GeoJson(
    simple,
    style_function=lambda x: {"color": "#8b949e", "weight": 1.5, "fillOpacity": 0,
                              "dashArray": "2,4"},
    name="cell buffer — approximate",
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds(exact_layer.get_bounds())
m

## 4. Cost against parent resolution

Coarser parents have longer boundaries, so the union gets more expensive while the cell
buffer stays flat — that gap is the reason the hull mode exists.

In [ ]:
# Fixed target resolution, so coarser parents really do have longer boundaries
TARGET = 12
print(f"{'parent res':>10} | {'cells on boundary':>17} | {'union (ms)':>10} | {'cell buffer (ms)':>16}")
print("-" * 64)

for res in range(4, 9):
    p = h3.latlng_to_cell(37.7759, -122.4180, res)
    target = TARGET

    start = time.perf_counter()
    b = h3b.get_buffered_boundary_polygon(p, intermediate_res=target)
    union_ms = (time.perf_counter() - start) * 1000

    start = time.perf_counter()
    h3b.get_buffered_h3_polygon(p)
    simple_ms = (time.perf_counter() - start) * 1000

    n = b["properties"]["num_boundary_cells"]
    print(f"{res:>10} | {n:>17,} | {union_ms:>10.2f} | {simple_ms:>16.2f}")

## Which to use

| You need | Use |
|---|---|
| A polygon that provably contains every descendant | `get_buffered_boundary_polygon(cell, res)` |
| The same, but fast and allowed to be generous | `get_buffered_boundary_polygon_cpp(cell, res, use_convex_hull=True)` |
| Just the exact boundary, no margin | `cell_boundary_from_children(cell, res)` |
| A rough outline of one cell, cheaply | `get_buffered_h3_polygon(cell)` |

Pass `buffer_meters` to any of them to set the margin yourself; `None` picks a safe default,
`0` disables buffering.